# Does `ft_api.py` work?

Validates the extracted fine-tuning harness before any real experiment depends on it. The durable output
is `runs/ft_*.json`, which the last cell copies to Drive.

Four things, cheapest first:

1. **Loaders** — split sizes match the published ones
2. **Truncation** — answered as a side effect (no GPU, no training)
3. **Acceptance test** — reproduce the pre-extraction numbers from `POC_v4_factory.ipynb`
   (XLM-R 0.127, mmBERT 0.537 on SIB-200). If this fails, the extraction is wrong and nothing
   built on it counts.
4. **Records** — `ft.table()` reads back what was written

Runtime: sections 1–3 are a few minutes. The NER section is 20–40 min and is behind a flag.

In [1]:
import os, sys
REPO = '/content/WashingtonCsed504'
FORK = 'https://github.com/patlkwok/WashingtonCsed504.git'   # YOUR fork, not upstream
if not os.path.exists(REPO):
    !git clone -q {FORK} {REPO}
sys.path.insert(0, f'{REPO}/src/a2-nlp')
import session; factory = session.start(prepare=False)   # no corpus: fine-tuning only

python 3.12.13 | NVIDIA A100-SXM4-80GB (85 GB, sm_80) | bf16: True
  installing seqeval ...
ready — cwd /content/WashingtonCsed504/src/a2-nlp


`session.start()` runs `git pull`, so a freshly pushed `ft_api.py` arrives here. If the next cell
raises `ModuleNotFoundError`, it was not pushed to the **fork** — the runtime only ever sees the
fork. If you edit `ft_api.py` and re-push, `importlib.reload(ft)` picks it up; a live kernel will
not re-import on its own, and `%autoreload` is the thing that dies on Python 3.12.

In [2]:
import importlib
import ft_api as ft
importlib.reload(ft)

print('ft_api', ft.API_VERSION, '| factory', factory.API_VERSION)
print('records ->', ft.RUNS)
print('defaults: sib', ft.FT_STEPS, 'steps | ner', ft.NER_STEPS, 'steps | batch', ft.FT_BATCH)

ft_api (1, 3) | factory (1, 0)
records -> /content/WashingtonCsed504/src/a2-nlp/runs
defaults: sib 352 steps | ner 2150 steps | batch 16


## 1. Loaders

In [3]:
sib = ft.load_sib200('yor_Latn')
ner = ft.load_masakhaner('yor')

sib_sizes = [len(sib[s]['text']) for s in ('train', 'validation', 'test')]
ner_sizes = [len(ner[s]['tokens']) for s in ('train', 'validation', 'test')]
assert sib_sizes == [701, 99, 204], sib_sizes
assert ner_sizes == [6876, 983, 1964], ner_sizes
print('OK — both match the published splits, so the Table 4 comparison is valid')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


README.md:   0%|          | 0.00/47.9k [00:00<?, ?B/s]

train.tsv:   0%|          | 0.00/128k [00:00<?, ?B/s]

dev.tsv:   0%|          | 0.00/17.0k [00:00<?, ?B/s]

test.tsv:   0%|          | 0.00/37.6k [00:00<?, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

sib200/yor_Latn: train 701 validation 99 test 204 | 7 classes (chance 0.143) | 1.1% decomposed -> NFC
masakhaner2/yor: train 6876 validation 983 test 1964 | 9 tags ['O', 'B-DATE', 'B-LOC', 'B-ORG', 'B-PER', 'I-DATE', 'I-LOC', 'I-ORG', 'I-PER']
  17.2% of characters are decomposed -> normalising to NFC  (see the module docstring -- this one matters)
OK — both match the published splits, so the Table 4 comparison is valid


## 2. Does truncation actually happen?

The tokenizer finding is currently framed as *"65% of the context window spent on fragments"*.
That is a claim about **truncation**. If `>128` is ~0% for both tokenizers, nothing is being
truncated, the context window is not the mechanism, and the penalty has to hurt through
representation quality instead — a different claim needing different evidence.

Lengths include the 2 special tokens both tokenizers add, so this is like-for-like.

In [4]:
XLMR = 'FacebookAI/xlm-roberta-base'
OWN  = 'tokenizers/yor-bpe16k'

print(f"{'dataset':<12}{'tokenizer':<20}{'mean':>7}{'p95':>6}{'p99':>6}{'max':>6}"
      f"{'>128':>7}{'tok/word':>10}")
print('-' * 74)
for name, kw in (('sib200',     dict(texts=sib['train']['text'] + sib['test']['text'])),
                 ('masakhaner', dict(words=ner['train']['tokens'] + ner['test']['tokens']))):
    for spec in (OWN, XLMR):
        r = ft.token_lengths(spec, max_length=ft.MAX_LEN, **kw)
        print(f"{name:<12}{spec.split('/')[-1]:<20}{r['mean']:>7.1f}{r['p95']:>6.0f}"
              f"{r['p99']:>6.0f}{r['max']:>6}{r['frac_over']:>7.1%}{r['tokens_per_word']:>10.2f}")

dataset     tokenizer              mean   p95   p99   max   >128  tok/word
--------------------------------------------------------------------------


[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (170 > 128). Running this sequence through the model will result in indexing errors


sib200      yor-bpe16k             44.1    72    95   170   0.1%      1.77


config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

...

sib200      xlm-roberta-base       70.4   124   154   225   4.0%      2.83


[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (144 > 128). Running this sequence through the model will result in indexing errors


masakhaner  yor-bpe16k             41.9    84   105   176   0.2%      1.67


[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (138 > 128). Running this sequence through the model will result in indexing errors


masakhaner  xlm-roberta-base       72.9   143   176   274  13.3%      2.91


## 3. Acceptance test — do the `POC_v4_factory.ipynb` numbers reproduce?

`FT_STEPS = 352` is exactly what that notebook's 8 epochs spent on the full 701-example split at
batch 16, so these two rows should land where they landed before. A difference below **0.06**
macro-F1 is not claimable on 204 test items — the test set does not resolve it — so that is the
threshold used here: inside it, the extraction is behaviour-preserving; outside it, something
moved.

In [5]:
REUSE = True   # False to rerun and rewrite the records with ft_api 1.3's chance/degenerate fields

BASELINES = {'XLM-R base': 'FacebookAI/xlm-roberta-base',
             'mmBERT base': 'jhu-clsp/mmBERT-base'}
EXPECTED_SIB = {'XLM-R base': 0.127, 'mmBERT base': 0.537}

sib_rows = {label: ft.evaluate(path, task='sib200', label=label, data=sib, reuse=REUSE)
            for label, path in BASELINES.items()}

chance = 1 / len(sib['labels'])
print(f'\nchance = {chance:.3f} ({len(sib["labels"])} balanced classes, '
      f'{len(sib["test"]["text"])} test items)\n')
print(f'{"model":<14}{"now":>7}{"sd":>7}{"notebook":>10}{"delta":>8}   verdict')
for label, rec in sib_rows.items():
    d = rec['mean'] - EXPECTED_SIB[label]
    # .get() so records written before ft_api 1.3 still read
    degenerate = rec.get('degenerate', rec['mean'] <= chance)
    if degenerate:
        v = 'DEGENERATE — never learned the task; reproduction unjudgeable'
    elif abs(d) < max(0.06, 2 * rec['sd']):
        v = 'reproduces'
    else:
        v = 'INVESTIGATE'
    print(f'{label:<14}{rec["mean"]:>7.3f}{rec["sd"]:>7.3f}{EXPECTED_SIB[label]:>10.3f}'
          f'{d:>+8.3f}   {v}')
    print(f'{"":14}per-seed {[round(s, 3) for s in rec["scores"]]}'
          f'   ft_api {tuple(rec.get("ft_api_version", ()))}')


  XLM-R base                 0.073 (reusing record; reuse=False to rerun)
  mmBERT base                0.529 (reusing record; reuse=False to rerun)

chance = 0.143 (7 balanced classes, 204 test items)

model             now     sd  notebook   delta   verdict
XLM-R base      0.073  0.022     0.127  -0.054   DEGENERATE — never learned the task; reproduction unjudgeable
              per-seed [0.057, 0.057, 0.105]   ft_api (1, 3)
mmBERT base     0.529  0.021     0.537  -0.008   reproduces
              per-seed [0.523, 0.507, 0.558]   ft_api (1, 3)


XLM-R at 0.127 is a question for the LR sweep, not a bug: a model scoring 0.843 on Yoruba NER cannot have
no usable Yoruba, so this is most likely a fine-tuning failure on 701 examples. Reproducing it
here is the point — `exp_xlmr_lr_sweep.ipynb` then sweeps LR **and** step budget against this baseline.

## 4. MasakhaNER — slower, behind a flag

In [6]:
RUN_NER = True   # ~20-40 min: 2150 steps x 3 seeds x 2 models, plus the bootstrap

EXPECTED_NER = {'XLM-R base': 0.843, 'mmBERT base': 0.848}
if RUN_NER:
    for label, path in BASELINES.items():
        rec = ft.evaluate(path, task='masakhaner', label=label, data=ner, reuse=REUSE)
        d = rec['mean'] - EXPECTED_NER[label]
        print(f'    vs notebook {EXPECTED_NER[label]:.3f} -> {d:+.3f}')
else:
    print('skipped — set RUN_NER = True')

  XLM-R base                 0.841 (reusing record; reuse=False to rerun)
    vs notebook 0.843 -> -0.002
  mmBERT base                0.851 (reusing record; reuse=False to rerun)
    vs notebook 0.848 -> +0.003


## 5. Records read back

In [7]:
rows = ft.table()
print(f'\n{len(rows)} record(s) in {ft.RUNS}')

model                     task          n_train      lr  steps  norm   score     sd   95% CI
mmBERT base               masakhaner       6876   3e-05   2150   NFC   0.851  0.007   [0.837, 0.865]
XLM-R base                masakhaner       6876   3e-05   2150   NFC   0.841  0.008   [0.827, 0.856]
mmBERT base               sib200            701   2e-05    352   NFC   0.529  0.021   [0.461, 0.581]
random init               sib200            701   5e-05    352   NFC   0.107  0.011   [0.086, 0.128]
XLM-R base                sib200            701   2e-05    352   NFC   0.073  0.022   [0.062, 0.084]

5 record(s) in /content/WashingtonCsed504/src/a2-nlp/runs


## 6. Optional — the from-scratch rows

The other two rows of the downstream table need checkpoints, and **checkpoints are not in git**
(hundreds of MB, deliberately untracked). So a fresh runtime has the two HF baselines and nothing
else. This cell prepares the Yoruba corpus and builds the random-init control, which is cheap; a
real from-scratch row needs either a checkpoint copied from Drive or a short pretraining run.

In [8]:
RUN_SCRATCH = True   # prepares the corpus (several minutes) before anything trains

if RUN_SCRATCH:
    factory = session.start(corpus='yor')       # verifies tokenizer fingerprint 15abd33de5af
    rand = factory.random_init('yor')
    ft.evaluate(rand, task='sib200', lr=ft.FT_LR_SCRATCH, label='random init',
                data=sib, reuse=REUSE)

    # A from-scratch row needs a pretrained checkpoint. The ones already measured are on Jeffrey's
    # workstation, so either copy one in from Drive, or train a small cell here:
    # rec = factory.pretrain('yor', tokens=32_000_000, steps=12_000)
    # ft.evaluate(os.path.join(factory.RUNS, rec['tag']), task='sib200',
    #             lr=ft.FT_LR_SCRATCH, label='from-scratch 33.8M', data=sib)
else:
    print('skipped — set RUN_SCRATCH = True')

python 3.12.13 | NVIDIA A100-SXM4-80GB (85 GB, sm_80) | bf16: True
  packages already present
yor: preparing yor_Latn
  using the shared tokenizer at 'tokenizers/yor-bpe16k' (not training a new one)


README.md:   0%|          | 0.00/329k [00:00<?, ?B/s]

    [fineweb2] 79,999 docs / 260M chars in 60s                        
    encoded 79,999 docs -> 69,596,452 tokens in 66s                        

  decoded sample: '<s> Ẹ̀gbá\nÀwọn àkóónú\nILẸ̀ Ẹ̀GBÁ[àtúnṣe | edit source]\nÓ se pàtàkì láti mọ díẹ̀ nípa ìtàn ilẹ̀ Ẹ̀gbá àti irú ènìyàn tí ń gbé ìlú Ẹ̀gbá. Ìdí èyí ni pé yóò jẹ́'
  69,096,452 train + 500,000 val tokens, 3.73 chars/token
  vocabulary fingerprint 15abd33de5af -- runs only compare across matching fingerprints

  corpus 'yor' ready, tokenizer 15abd33de5af
ready — cwd /content/WashingtonCsed504/src/a2-nlp


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  random init                0.107 (reusing record; reuse=False to rerun)


## 7. Save

`runs/` is on the runtime's disposable disk. These are the project's **first** downstream
records — losing them costs the whole session.

In [9]:
session.save_results()   # asserts Drive is really mounted before copying

Mounted at /content/drive
copied 143 result files -> /content/drive/MyDrive/csed504-runs


143